
#GroupDNA - WhatsApp Chat Analyzer


**Name**:Vedant

**Batch**:DS(September 2026)

**Date**:2026-9-20

In [65]:
from datetime import datetime, timedelta
import numpy as np
import string
import os
import json

#FEATURE 1: THE CHAT PARSER

In [66]:
filename = 'hostel_bois.txt'

try:
    with open(filename, 'r', encoding='utf-8') as file:
        raw_lines = file.readlines()
except FileNotFoundError:
    print(f"Error: '{filename}' not found. Please upload it first.")
    raw_lines = []

parsed_chat_data = []
skipped_system_logs = 0
count_media = 0
count_deleted = 0

last_valid_msg = None

for line in raw_lines:
    line = line.strip()
    if not line:
        continue

    parts = line.split(' - ', 1)
    is_valid_new_line = False

    if len(parts) == 2:
        time_part, rest = parts
        try:
            try:
                dt_obj = datetime.strptime(time_part, '%d/%m/%y, %H:%M')
            except ValueError:
                dt_obj = datetime.strptime(time_part, '%d/%m/%Y, %H:%M')
            is_valid_new_line = True
        except ValueError:
            is_valid_new_line = False

    if not is_valid_new_line:
        if last_valid_msg is not None:
            parsed_chat_data[-1]['message_body'] += " " + line
        continue

    sender_split = rest.split(': ', 1)

    if len(sender_split) < 2:
        skipped_system_logs += 1
        continue

    sender_name, msg_content = sender_split

    if msg_content.endswith('<This message was edited>'):
        msg_content = msg_content.replace('<This message was edited>', '').strip()

    media_flag = False
    deleted_flag = False

    if msg_content == '<Media omitted>':
        count_media += 1
        media_flag = True
    elif msg_content == 'This message was deleted':
        count_deleted += 1
        deleted_flag = True

    msg_dict = {
        'datetime': dt_obj,
        'sender': sender_name,
        'message_body': msg_content,
        'is_media': media_flag,
        'is_deleted': deleted_flag
    }

    parsed_chat_data.append(msg_dict)
    last_valid_msg = msg_dict

def is_phone_number(name):
    digits_count = sum(1 for c in name if c.isdigit())
    if digits_count >= 10 or (name.startswith('+91') and digits_count >= 8):
        return True
    return False

phone_map = {}
fake_names = ["Rohan", "Aditya", "Kabir", "Arjun", "Aryan", "Vihaan", "Aarav", "Krishna", "Ishaan", "Shaurya", "Aarohi", "Ananya", "Diya", "Isha", "Kavya", "Kiara", "Meera", "Neha", "Pooja", "Riya", "Sneha", "Tara"]
user_counter = 0

for m in parsed_chat_data:
    s = m['sender']
    if is_phone_number(s):
        if s not in phone_map:
            if user_counter < len(fake_names):
                phone_map[s] = fake_names[user_counter]
            else:
                phone_map[s] = f"Guest_{user_counter}"
            user_counter += 1
        m['sender'] = phone_map[s]

unique_senders = set(m['sender'] for m in parsed_chat_data)
total_parsed = len(parsed_chat_data)

print(f"Successfully parsed {total_parsed} messages from {len(unique_senders)} participants.")
print(f"Skipped {skipped_system_logs} system messages, {count_media} media-omitted, and {count_deleted} deleted messages.")


Successfully parsed 3174 messages from 6 participants.
Skipped 4 system messages, 32 media-omitted, and 15 deleted messages.


# FEATURE 2: GROUP OVERVIEW


In [67]:
if parsed_chat_data:
    first_msg_date = parsed_chat_data[0]['datetime']
    last_msg_date = parsed_chat_data[-1]['datetime']
    total_active_days = (last_msg_date - first_msg_date).days + 1

    person_msg_count = {}
    for m in parsed_chat_data:
        sender = m['sender']
        person_msg_count[sender] = person_msg_count.get(sender, 0) + 1

    ranked_participants = sorted(person_msg_count.items(), key=lambda item: item[1], reverse=True)

    base_name = os.path.basename(filename).replace('.txt', '')
    if base_name.startswith('WhatsApp Chat with '):
        base_name = base_name.replace('WhatsApp Chat with ', '')
    if base_name == 'hostel_bois':
        base_name = 'Hostel bois 4ever'

    overview_data = {
        "group_name": base_name,
        "period_str": f"{first_msg_date.strftime('%d %B %Y')} to {last_msg_date.strftime('%d %B %Y')} ({total_active_days} days)",
        "total_messages": len(parsed_chat_data),
        "participant_count": len(ranked_participants),
        "ranked_counts": ranked_participants
    }



# FEATURE 3: MOST ACTIVE DAY AND HOUR

In [68]:
daily_activity = {}
hourly_activity = {}

for m in parsed_chat_data:
      msg_date = m['datetime'].date()
      msg_hour = m['datetime'].hour

      daily_activity[msg_date] = daily_activity.get(msg_date, 0) + 1
      hourly_activity[msg_hour] = hourly_activity.get(msg_hour, 0) + 1

peak_day = max(daily_activity, key=daily_activity.get)
peak_hour = max(hourly_activity, key=hourly_activity.get)

peak_day_msgs = daily_activity[peak_day]
avg_peak_hour_msgs_per_day = hourly_activity[peak_hour] / total_active_days

activity_data = {
      "peak_day": peak_day.strftime('%d %B %Y'),
      "peak_day_msgs": peak_day_msgs,
      "peak_hour_start": peak_hour,
      "avg_peak_hour_msgs": int(avg_peak_hour_msgs_per_day)
    }


 # FEATURE 4: ACTIVITY HEATMAP (NumPy)

In [69]:
top_n_names = [p[0] for p in ranked_participants[:9]]
activity_matrix = np.zeros((len(top_n_names), 24), dtype=int)

for m in parsed_chat_data:
        sender = m['sender']
        if sender in top_n_names:
            row_idx = top_n_names.index(sender)
            col_idx = m['datetime'].hour
            activity_matrix[row_idx, col_idx] += 1


 # FEATURE 5: TOP WORDS

In [70]:
ignore_words = {'i', 'is', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'on', 'for', 'it', 'my', 'me', 'you', 'that', 'this',
                    'hai', 'ki', 'se', 'ko', 'bhi', 'na', 'toh', 'ke', 'ka', 'ye', 'kya',
                    'was', 'how', 'so', 'about', 'am', 'at', 'he', 'his', 'with', 'but', 'are', 'not', 'have', 'from', 'they',
                    'all', 'we', 'do', 'what', 'be', 'just', 'like', 'there', 'if', 'out', 'up', 'when', 'who', 'an', 'as',
                    'can', 'would', 'them', 'some', 'were', 'their', 'has', 'will', 'our', 'been', 'which', 'had', 'no', 'did',
                    'one', 'then', 'now', 'by', 'she', 'her', 'your', 'about', 'why', 'ho', 'bhai', 'hi', 'nhi'}

global_word_freq = {}

for m in parsed_chat_data:
        if m['is_media'] or m['is_deleted']:
            continue

        text_content = m['message_body'].lower()

        for char in string.punctuation:
            text_content = text_content.replace(char, ' ')

        tokens = text_content.split()
        for word in tokens:
            if word not in ignore_words and len(word) > 1:
                global_word_freq[word] = global_word_freq.get(word, 0) + 1

top_10_words = sorted(global_word_freq.items(), key=lambda x: x[1], reverse=True)[:10]


# FEATURE 6: RESPONSE SPEED & SILENT STREAKS

In [71]:
person_gaps = {p: [] for p in top_n_names}

previous_sender = None
previous_time = None

for m in parsed_chat_data:
        sender = m['sender']
        if sender in top_n_names:
            if previous_sender and previous_sender != sender:
                time_difference = (m['datetime'] - previous_time).total_seconds()
                person_gaps[sender].append(time_difference)

        previous_sender = sender
        previous_time = m['datetime']

avg_response_speed = {}
for p, gaps in person_gaps.items():
        avg_response_speed[p] = (sum(gaps) / len(gaps)) if gaps else 0

all_chat_dates = sorted(list({m['datetime'].date() for m in parsed_chat_data}))
longest_streaks = {p: (0, None, None) for p in top_n_names}

for p in top_n_names:
        max_silent_days = 0
        current_silent = 0
        current_start = None
        best_start = None
        best_end = None

        active_dates = {m['datetime'].date() for m in parsed_chat_data if m['sender'] == p}

        for d in all_chat_dates:
            if d not in active_dates:
                if current_silent == 0:
                    current_start = d
                current_silent += 1
                if current_silent > max_silent_days:
                    max_silent_days = current_silent
                    best_start = current_start
                    best_end = d
            else:
                current_silent = 0

        longest_streaks[p] = (max_silent_days, best_start, best_end)

fastest_person = min([p for p in avg_response_speed.items() if p[1] > 0], key=lambda x: x[1])
slowest_person = max([p for p in avg_response_speed.items() if p[1] > 0], key=lambda x: x[1])



  # FEATURE 7: PERSONALITY ARCHETYPE DETECTION

In [72]:
raw_arch_scores = {p: {} for p in top_n_names}

    # 1. THE SPAMMER
consecutive_bursts = {p: [] for p in top_n_names}
current_burst_count = 0
active_sender = None

for m in parsed_chat_data:
        s = m['sender']
        if s in top_n_names:
            if s == active_sender:
                current_burst_count += 1
            else:
                if active_sender:
                    consecutive_bursts[active_sender].append(current_burst_count)
                active_sender = s
                current_burst_count = 1
        else:
            if active_sender:
                consecutive_bursts[active_sender].append(current_burst_count)
            active_sender = None
            current_burst_count = 0
if active_sender:
        consecutive_bursts[active_sender].append(current_burst_count)

for p in top_n_names:
        b_list = consecutive_bursts[p]
        raw_arch_scores[p]['THE SPAMMER'] = (sum(b_list) / len(b_list)) if b_list else 0

    # 2. THE GROUP MOM
mom_words = ['okay', 'safe', 'eat', 'sleep', 'take care', 'are you', 'please', 'reminder', 'drink water', "don't forget"]
for p in top_n_names:
        raw_arch_scores[p]['THE GROUP MOM'] = 0

for m in parsed_chat_data:
        if m['sender'] in top_n_names and not m['is_media'] and not m['is_deleted']:
            cleaned_text = m['message_body'].lower()
            for punct in string.punctuation:
                if punct != "'":
                    cleaned_text = cleaned_text.replace(punct, ' ')
            cleaned_text = " " + cleaned_text + " "

            for mw in mom_words:
                if f" {mw.replace(',', '').replace('.', '')} " in cleaned_text:
                    raw_arch_scores[m['sender']]['THE GROUP MOM'] += 1
                    break

    # 3. THE NIGHT OWL
for idx, p in enumerate(top_n_names):
        night_count = activity_matrix[idx, 23] + np.sum(activity_matrix[idx, 0:5])
        total_count = np.sum(activity_matrix[idx, :])
        raw_arch_scores[p]['THE NIGHT OWL'] = (night_count / total_count * 100) if total_count > 0 else 0

    # 4. THE STORYTELLER
word_lengths = {p: [] for p in top_n_names}
for m in parsed_chat_data:
        if m['sender'] in top_n_names and not m['is_media'] and not m['is_deleted']:
            word_lengths[m['sender']].append(len(m['message_body'].split()))

for p in top_n_names:
        raw_arch_scores[p]['THE STORYTELLER'] = (sum(word_lengths[p]) / len(word_lengths[p])) if word_lengths[p] else 0

    # 5. THE DRAMA QUEEN
for p in top_n_names:
        raw_arch_scores[p]['THE DRAMA QUEEN'] = 0

for m in parsed_chat_data:
        if m['sender'] in top_n_names and not m['is_media'] and not m['is_deleted']:
            msg_text = m['message_body']
            only_letters = ''.join([c for c in msg_text if c.isalpha()])
            if len(only_letters) >= 3 and only_letters.isupper():
                raw_arch_scores[m['sender']]['THE DRAMA QUEEN'] += 1

for p in top_n_names:
        raw_arch_scores[p]['THE DRAMA QUEEN'] = (raw_arch_scores[p]['THE DRAMA QUEEN'] / person_msg_count[p] * 100)

    # 6. THE GHOST
for p in top_n_names:
        days_participated = len({m['datetime'].date() for m in parsed_chat_data if m['sender'] == p})
        days_absent = total_active_days - days_participated
        raw_arch_scores[p]['THE GHOST'] = (days_absent / total_active_days * 100)

    # 7. THE COMEDIAN
haha_terms = ['lol', 'lmao', 'haha', 'rofl', 'lmfao', 'hehe', '😂', '🤣']
for p in top_n_names:
        raw_arch_scores[p]['THE COMEDIAN'] = 0

for m in parsed_chat_data:
        if m['sender'] in top_n_names and not m['is_media'] and not m['is_deleted']:
            text_lower = m['message_body'].lower()
            if any(h in text_lower for h in haha_terms):
                raw_arch_scores[m['sender']]['THE COMEDIAN'] += 1

for p in top_n_names:
        raw_arch_scores[p]['THE COMEDIAN'] = (raw_arch_scores[p]['THE COMEDIAN'] / person_msg_count[p] * 100)

    # 8. THE QUESTION MASTER
for p in top_n_names:
        raw_arch_scores[p]['THE QUESTION MASTER'] = 0

for m in parsed_chat_data:
        if m['sender'] in top_n_names and not m['is_media'] and not m['is_deleted']:
            if '?' in m['message_body']:
                raw_arch_scores[m['sender']]['THE QUESTION MASTER'] += 1

for p in top_n_names:
        raw_arch_scores[p]['THE QUESTION MASTER'] = (raw_arch_scores[p]['THE QUESTION MASTER'] / person_msg_count[p] * 100)

    # 9. THE MEDIA MOGUL
for p in top_n_names:
        raw_arch_scores[p]['THE MEDIA MOGUL'] = 0

for m in parsed_chat_data:
        if m['sender'] in top_n_names and m['is_media']:
            raw_arch_scores[m['sender']]['THE MEDIA MOGUL'] += 1

    # Exclusive Assignment Logic
archetype_keys = ['THE SPAMMER', 'THE GROUP MOM', 'THE NIGHT OWL', 'THE STORYTELLER', 'THE DRAMA QUEEN', 'THE GHOST', 'THE MEDIA MOGUL', 'THE COMEDIAN', 'THE QUESTION MASTER']
scaled_scores = {p: {} for p in top_n_names}

for arch in archetype_keys:
        highest_arch_val = max(raw_arch_scores[p][arch] for p in top_n_names)
        for p in top_n_names:
            scaled_scores[p][arch] = (raw_arch_scores[p][arch] / highest_arch_val) if highest_arch_val > 0 else 0

assigned_archetypes = {}
people_pool = set(top_n_names)

for a in archetype_keys:
        best_match = None
        max_relative_score = -1

        for p in people_pool:
            if scaled_scores[p][a] > max_relative_score:
                max_relative_score = scaled_scores[p][a]
                best_match = p

        if best_match:
            assigned_archetypes[a] = (best_match, raw_arch_scores[best_match][a])
            people_pool.remove(best_match)


 # FEATURE 8: THE FINAL REPORT

In [73]:
def generate_bar(val, max_val, max_length=20):
        length = int((val / max_val) * max_length)
        return '█' * length

max_name_len = max(len(p) for p in top_n_names)
if max_name_len < 15: max_name_len = 15

print("=" * 60)
print(f" GROUPDNA REPORT - \"{overview_data['group_name']}\"")
print(f" {total_active_days} days • {overview_data['total_messages']:,} messages • {overview_data['participant_count']} members")
print("=" * 60)

print(f" Period       : {first_msg_date.strftime('%d %B %Y')} to {last_msg_date.strftime('%d %B %Y')}")
print(f" Busiest day  : {activity_data['peak_day']} ({activity_data['peak_day_msgs']} messages)")
print(f" Busiest hour : {activity_data['peak_hour_start']:02d}:00 - {activity_data['peak_hour_start']+1:02d}:00")

print(f"\n MESSAGES PER PERSON")
highest_msgs = ranked_participants[0][1]
for person, count in ranked_participants[:9]:
        percentage = (count / overview_data['total_messages']) * 100
        bar_str = generate_bar(count, highest_msgs)
        if count < 100:
            bar_str = "."
        print(f" {person:<{max_name_len}} {bar_str:<20} {count} ({percentage:.1f}%)")

print("\n ACTIVITY HEATMAP (hour of day, columns 00 to 23)")
print(" " + " " * max_name_len + "00 03 06 09 12 15 18 21")

for i, p in enumerate(top_n_names):
        row_data = activity_matrix[i]
        max_val = max(row_data) if max(row_data) > 0 else 1

        blocks = []
        for val in row_data:
            pct = val / max_val
            if pct == 0 or pct <= 0.25:
                blocks.append(' ')
            elif pct <= 0.50:
                blocks.append('░')
            elif pct <= 0.75:
                blocks.append('▒')
            else:
                blocks.append('█')

        annotation = ""
        if 'THE NIGHT OWL' in assigned_archetypes and assigned_archetypes['THE NIGHT OWL'][0] == p:
            annotation = " <- NIGHT OWL"
        print(f" {p:<{max_name_len}} " + "".join(blocks) + annotation)

print("\n THIS GROUP'S FAVOURITE WORDS")
if top_10_words:
        top_word_count = top_10_words[0][1]
        for word, count in top_10_words:
            bar_str = generate_bar(count, top_word_count)
            print(f" {word:<10} {bar_str:<20} {count}")

print("\n RESPONSE PATTERNS")
f_person = fastest_person[0]
s_person = slowest_person[0]
print(f" Fastest replier : {f_person} (avg {fastest_person[1]/60:.1f} minutes)")
print(f" Slowest replier : {s_person} (avg {slowest_person[1]/3600:.1f} hours)")

print("\n LONGEST SILENT STREAKS")
streak_count = 0
for p, (streak_days, start_d, end_d) in sorted(longest_streaks.items(), key=lambda x: x[1][0], reverse=True):
        if streak_days > 0:
            if streak_count == 0 and start_d and end_d:
                date_str = f" ({start_d.strftime('%d %b')} - {end_d.strftime('%d %b')})"
            else:
                date_str = ""
            print(f" {p:<{max_name_len}} : {streak_days} days{date_str}")
            streak_count += 1

print("\n PERSONALITY ARCHETYPES")
for arch in archetype_keys:
        if arch in assigned_archetypes:
            p, score = assigned_archetypes[arch]

            reason = f"score: {score:.1f}"
            if arch == 'THE SPAMMER': reason = f"avg {score:.1f} msgs in a row"
            elif arch == 'THE NIGHT OWL': reason = f"{score:.1f}% msgs between 23h-04h"
            elif arch == 'THE STORYTELLER': reason = f"avg {score:.1f} words per msg"
            elif arch == 'THE DRAMA QUEEN': reason = f"{score:.1f}% ALL-CAPS messages"
            elif arch == 'THE GHOST': reason = f"silent on {int(score * total_active_days / 100)} of {total_active_days} days"
            elif arch == 'THE GROUP MOM': reason = f"caring keyword score: {int(score)}"
            elif arch == 'THE MEDIA MOGUL': reason = f"sent {int(score)} media files"
            elif arch == 'THE COMEDIAN': reason = f"{score:.1f}% haha/lol messages"
            elif arch == 'THE QUESTION MASTER': reason = f"{score:.1f}% messages with ?"

            print(f" {p:<{max_name_len}} → {arch:<19} ({reason})")

print("=" * 60)
print(" Generated by GroupDNA • Built with Python + NumPy")
print("=" * 60)

 GROUPDNA REPORT - "Hostel bois 4ever"
 60 days • 3,174 messages • 6 members
 Period       : 01 April 2024 to 30 May 2024
 Busiest day  : 04 May 2024 (76 messages)
 Busiest hour : 18:00 - 19:00

 MESSAGES PER PERSON
 Rahul           ████████████████████ 953 (30.0%)
 Priya           ███████████████      718 (22.6%)
 Neha            █████████████        635 (20.0%)
 Aman            ██████████           490 (15.4%)
 Karan           ███████              354 (11.2%)
 Vikas           .                    24 (0.8%)

 ACTIVITY HEATMAP (hour of day, columns 00 to 23)
                00 03 06 09 12 15 18 21
 Rahul                       ▒░░▒▒░█▒░█▒▒
 Priya                  ░▒████▒▒░░▒▒█▒░░ 
 Neha                 ░  ▒██░▒▒░ ▒███▒░░░
 Aman            ▒█▒▒█                  ▒ <- NIGHT OWL
 Karan                   ░░▒░█▒█▒▒▒▒█▒░  
 Vikas                  ░█░░ ▒▒ ░░█▒▒░░░▒

 THIS GROUP'S FAVOURITE WORDS
 guys       ████████████████████ 318
 today      ██████████████████   292
 everyone   ████████████ 

# FEATURE 9: DASHBOARD JSON EXPORTER

In [74]:
dashboard_payload = {
        "group_name": overview_data['group_name'],
        "metrics": {
            "total_messages": overview_data['total_messages'],
            "total_members": overview_data['participant_count'],
            "active_days": total_active_days,
            "busiest_day": f"{activity_data['peak_day']} ({activity_data['peak_day_msgs']} msgs)",
            "busiest_hour": f"{activity_data['peak_hour_start']:02d}:00 - {activity_data['peak_hour_start']+1:02d}:00"
        },
        "messages_per_person": [
            {
                "name": person,
                "count": count,
                "percentage": round((count / overview_data['total_messages']) * 100, 1)
            }
            for person, count in ranked_participants[:9]
        ],
        "activity_heatmap": {
            "participants": top_n_names,
            "matrix": activity_matrix.tolist()
        },
        "top_words": [
            {"word": word, "count": count} for word, count in top_10_words
        ],
        "response_patterns": {
            "fastest_replier": {
                "name": fastest_person[0],
                "avg_minutes": round(fastest_person[1] / 60, 1)
            },
            "slowest_replier": {
                "name": slowest_person[0],
                "avg_hours": round(slowest_person[1] / 3600, 1)
            }
        },
        "silent_streaks": [
            {
                "name": p,
                "days": streak_days,
                "start": start_d.strftime('%Y-%m-%d') if start_d else None,
                "end": end_d.strftime('%Y-%m-%d') if end_d else None
            }
            for p, (streak_days, start_d, end_d) in sorted(longest_streaks.items(), key=lambda x: x[1][0], reverse=True)
            if streak_days > 0
        ],
        "personality_archetypes": [
            {
                "archetype": arch,
                "person": assigned_archetypes[arch][0],
                "score": round(assigned_archetypes[arch][1], 1)
            }
            for arch in archetype_keys if arch in assigned_archetypes
        ]
    }

    # Write dashboard JSON file
with open('groupdna_data.json', 'w', encoding='utf-8') as json_file:
      json.dump(dashboard_payload, json_file, indent=4)
print("\n[Dashboard] Successfully generated 'groupdna_data.json' for Web UI integration.")


[Dashboard] Successfully generated 'groupdna_data.json' for Web UI integration.


## Final Reflection
**The Hardest Part:**
Parsing the datetime strings and tracking response gaps was surprisingly tricky. Real-world data is extremely messy - having to account for system messages, deleted texts, and missing media files proved that data cleaning is easily 80% of the job! Additionally, structuring the logic for the Activity Heatmap array and dynamically tracking the longest silent streaks without using Pandas required a lot of careful thought.

**What I'd Do Differently:**
If I had more time, I would try to implement a more robust regex-based parser to handle multi-line messages flawlessly. I'd also love to do sentiment analysis to see who is the most positive or negative in the group.

**My Archetype (Personal Chat Bonus):**
When I ran this tool on my actual college group chat, I found out I am **THE DRAMA QUEEN** (with ALL-CAPS messages)! Even though I was an active member, seeing the raw stats for myself and the rest of the group felt strangely personal. The data was painful but very real... numbers don't lie! 😅